In [0]:
%sql
-- Infrastructure Setup: Creating the Medallion Architecture layers
CREATE CATALOG IF NOT EXISTS globalmart;
USE CATALOG globalmart;

CREATE SCHEMA IF NOT EXISTS bronze
COMMENT 'Raw data ingested directly from source systems; immutable and enriched with metadata.';

CREATE SCHEMA IF NOT EXISTS silver
COMMENT 'Cleaned, conformed, and filtered data; ready for analysis and joining.';

CREATE SCHEMA IF NOT EXISTS gold
COMMENT 'Aggregated, business-level datasets and reporting tables.';

CREATE SCHEMA IF NOT EXISTS metadata
COMMENT 'Operational tracking, ingestion logs, and schema evolution history.';

-- Verification of the bronze layer setup
DESCRIBE SCHEMA EXTENDED bronze;

In [0]:
%sql
ALTER TABLE globalmart.metadata.ingestion_log ADD COLUMN target_table STRING

In [0]:
import re
from pyspark.sql.functions import current_timestamp, lit, col

def ingest_file_idempotently(file_path, target_table):
    # Path cleaning for Unity Catalog compatibility
    clean_path = file_path.replace("dbfs:", "")
    file_name = clean_path.split("/")[-1]
    
    log_table = "globalmart.metadata.ingestion_log"
    bronze_table = f"globalmart.bronze.{target_table}"
    
    # Check if this file has already been successfully processed
    already_done = spark.table(log_table) \
        .filter((col("file_name") == file_name) & (col("status") == "SUCCESS")) \
        .count() > 0
    
    if already_done:
        print(f"Skipping {file_name}: Already processed.")
        return

    try:
        print(f"Ingesting: {file_name} -> {bronze_table}")
        
        # Read the raw CSV
        df = (spark.read.format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .load(clean_path)
              .withColumn("ingested_at", current_timestamp())
              .withColumn("source_file", lit(file_name)))
        
        # Write to the Bronze layer
        df.write.mode("append").saveAsTable(bronze_table)
        
        # Log success with explicit column mapping
        spark.sql(f"""
            INSERT INTO {log_table} (file_name, target_table, ingested_at, status)
            VALUES ('{file_name}', '{target_table}', current_timestamp(), 'SUCCESS')
        """)
        print(f"Successfully loaded {target_table}")
        
    except Exception as e:
        print(f"Error ingesting {file_name}: {str(e)}")
        # Log failure with explicit column mapping
        spark.sql(f"""
            INSERT INTO {log_table} (file_name, target_table, ingested_at, status)
            VALUES ('{file_name}', '{target_table}', current_timestamp(), 'FAILED')
        """)

# --- Execution Section ---

# 1. Scan the folder for raw data
raw_files = dbutils.fs.ls("/Volumes/globalmart/bronze/raw_data/")

print(f"{'Detected File':<50} | Status")
print("-" * 75)

for f in raw_files:
    # 2. Only process CSV files
    if not f.name.endswith(".csv"):
        continue
        
    # 3. Clean filename to get table name (e.g., olist_customers_dataset.csv -> customers)
    t_name = re.sub(r'(olist_|_dataset|.csv)', '', f.name)
    
    # 4. Execute ingestion
    try:
        ingest_file_idempotently(f.path, t_name)
    except Exception as e:
        print(f"Loop Error with {f.name}: {str(e)[:60]}")

print("-" * 75)
print("Ingestion Process Finished.")

In [0]:
# Create a summary of all ingested bronze tables
import pandas as pd

report_list = []
# Get all tables in the bronze schema
tables = spark.sql("SHOW TABLES IN globalmart.bronze").collect()

for row in tables:
    t_name = row['tableName']
    count = spark.table(f"globalmart.bronze.{t_name}").count()
    cols = len(spark.table(f"globalmart.bronze.{t_name}").columns)
    report_list.append({"Table": t_name, "Rows": count, "Columns": cols})

# Display as a neat table
display(spark.createDataFrame(report_list))




In [0]:
%sql
-- Final Check for Task 1.2
SELECT 
    table_name, 
    table_catalog, 
    table_schema, 
    'SUCCESS' as status 
FROM globalmart.information_schema.tables 
WHERE table_schema = 'bronze';

In [0]:
%sql
SELECT 
    table_name, 
    count(*) AS record_count,
    'SUCCESS' AS status
FROM (
    SELECT 'customers' as table_name FROM globalmart.bronze.customers UNION ALL
    SELECT 'geolocation' FROM globalmart.bronze.geolocation UNION ALL
    SELECT 'order_items' FROM globalmart.bronze.order_items UNION ALL
    SELECT 'order_payments' FROM globalmart.bronze.order_payments UNION ALL
    SELECT 'order_reviews' FROM globalmart.bronze.order_reviews UNION ALL
    SELECT 'orders' FROM globalmart.bronze.orders UNION ALL
    SELECT 'products' FROM globalmart.bronze.products UNION ALL
    SELECT 'sellers' FROM globalmart.bronze.sellers UNION ALL
    -- Use the full name found in your bronze database
    SELECT 'category_translation' FROM globalmart.bronze.product_category_name_translation
)
GROUP BY table_name;

In [0]:
# 1. Scan the Volume
raw_files = dbutils.fs.ls("/Volumes/globalmart/bronze/raw_data/")

print(f"{'Processing File':<50} | Status")
print("-" * 80)

for f in raw_files:
    # Skip non-csv files (like metadata or folders)
    if not f.name.endswith(".csv"):
        continue

    # Get the system path and strip 'dbfs:' for Unity Catalog
    path = f.path.replace("dbfs:", "")
    
    # IMPROVED NAMING: 
    # This handles both 'olist_orders_dataset.csv' AND 'product_category_name_translation.csv'
    t_name = f.name.replace("olist_", "").replace("_dataset", "").replace(".csv", "")
    
    try:
        # This calls your idempotent function
        ingest_file_idempotently(path, t_name)
    except Exception as e:
        # Using a slightly wider slice to see more error detail if it happens
        print(f"{f.name:<50} | Error: {str(e)[:100]}")

print("-" * 80)
print("Ingestion Process Finished.")

In [0]:
# Specifically hunt for the translation file
all_files = dbutils.fs.ls("/Volumes/globalmart/bronze/raw_data/")
# Find the path and clean it for Spark compatibility
translation_file = [f.path.replace("dbfs:", "") for f in all_files if "translation" in f.name.lower()]

if translation_file:
    print(f"Found it: {translation_file[0]}")
    # Force the clean name for your SQL reports
    ingest_file_idempotently(translation_file[0], "category_translation")
else:
    print("File not found. Please check if 'product_category_name_translation.csv' is uploaded.")

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS globalmart.bronze.checkpoints;

In [0]:
# 1. Setup paths
source_dir = "/Volumes/globalmart/bronze/raw_data/"
# We use a unique subfolder for this specific task
checkpoint_path = "/Volumes/globalmart/bronze/checkpoints/task_1_3_orders"

def run_orders_autoloader():
    query = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
        .option("header", "true")
        # Watching the directory, but filtering for the specific orders file
        .load(source_dir)
        .filter("_metadata.file_name LIKE '%olist_orders_dataset.csv%'") 
        .writeStream
        .option("checkpointLocation", f"{checkpoint_path}/check")
        .trigger(availableNow=True)
        .toTable("globalmart.bronze.orders_autoloader"))
    
    query.awaitTermination()

# Run 1
print("Running Ingestion 1...")
run_orders_autoloader()
count1 = spark.table("globalmart.bronze.orders_autoloader").count()
print(f"Run 1 Total Rows: {count1}")

# Run 2 (Demonstrating Idempotency)
print("Running Ingestion 2...")
run_orders_autoloader()
count2 = spark.table("globalmart.bronze.orders_autoloader").count()
print(f"Run 2 Total Rows: {count2}")

In [0]:
from pyspark.sql import functions as F

# 1. Load the Bronze Payments data
df_payments = spark.table("globalmart.bronze.order_payments")

# 2. Create the Nested Table
# We use collect_list to turn multiple payment rows into one array per order
nested_df = (df_payments
    .groupBy("order_id")
    .agg(
        F.collect_list(
            F.struct("payment_type", "payment_installments", "payment_value")
        ).alias("payment_details"),
        F.sum("payment_value").alias("total_payment_value"),
        F.count("payment_sequential").alias("num_payment_methods")
    ))

nested_df.write.mode("overwrite").saveAsTable("globalmart.bronze.payments_nested")

# 3. Create the Flattened Table
# We use explode to prove we can unpack the nested data back to its original grain
flattened_df = (nested_df
    .withColumn("payment", F.explode("payment_details"))
    .select("order_id", "total_payment_value", "payment.*"))

flattened_df.write.mode("overwrite").saveAsTable("globalmart.bronze.payments_flattened")

# 4. Generate the Final Milestone Report
max_methods = nested_df.select(F.max("num_payment_methods")).collect()[0][0]
multi_pay_count = nested_df.filter("num_payment_methods > 1").count()
total_orders = nested_df.count()
percentage = (multi_pay_count / total_orders) * 100

print("--- FINAL MILESTONE 1 REPORT ---")
print(f"Nested Count ({nested_df.count()}) == Distinct Orders ({df_payments.select('order_id').distinct().count()})")
print(f"Flattened Count ({flattened_df.count()}) == Original Row Count ({df_payments.count()})")
print(f"Max payment methods used: {max_methods}")
print(f"Orders with >1 payment method: {percentage:.2f}%")

In [0]:
import pandas as pd
from pyspark.sql import functions as F

# 1. Read existing data to modify it
orders_path = "/Volumes/globalmart/bronze/raw_data/olist_orders_dataset.csv"
pdf = pd.read_csv(orders_path)

# 2. Add two new categorical columns
# We'll populate them only for the first 100 rows to simulate "New Data"
pdf['order_priority'] = None
pdf['is_gift'] = None

pdf.loc[0:99, 'order_priority'] = 'High'
pdf.loc[0:99, 'is_gift'] = 'Yes'

# 3. Save as a new version in the Volume
new_source_path = "/Volumes/globalmart/bronze/raw_data/olist_orders_v2.csv"
pdf.to_csv(new_source_path, index=False)

print(f"New version created at: {new_source_path}")

In [0]:
from pyspark.sql import functions as F

# 1. Read the new file (V2) 
# We use inferSchema=True so it matches the 'timestamp' type of your current table
df_new = (spark.read
          .option("header", "true")
          .option("inferSchema", "true") 
          .csv(new_source_path))

# 2. Add the two new columns for Task 1.5
df_new = (df_new
          .withColumn("order_priority", F.lit("High"))
          .withColumn("is_gift", F.lit("Yes"))
          .limit(100))

# 3. FORCE the merge
# This will resolve the 'Failed to merge fields' error by resetting the metadata
(df_new.write
  .format("delta")
  .mode("append")
  .option("mergeSchema", "true") 
  .option("overwriteSchema", "true") 
  .saveAsTable("globalmart.bronze.orders"))

print("Success! The Bronze table has evolved and the type conflict is resolved.")

In [0]:
from pyspark.sql import functions as F

def validate_schema(df, expected_schema, table_name):
    """
    Compares a DataFrame's schema against an expected schema contract.
    Logs missing columns, extra columns, and type mismatches.
    """
    actual_schema = df.schema
    violations = []
    
    # Create dictionaries of {name: type} for comparison
    expected_cols = {field.name: str(field.dataType) for field in expected_schema}
    actual_cols = {field.name: str(field.dataType) for field in actual_schema}
    
    # 1. Check for missing columns or type mismatches
    for col, expected_dtype in expected_cols.items():
        if col not in actual_cols:
            violations.append({"table": table_name, "issue": "Missing Column", "detail": col})
        elif actual_cols[col] != expected_dtype:
            violations.append({"table": table_name, "issue": "Type Mismatch", 
                               "detail": f"{col}: expected {expected_dtype}, got {actual_cols[col]}"})
            
    # 2. Check for extra columns (this is where Task 1.5 will show up!)
    for col in actual_cols:
        if col not in expected_cols:
            violations.append({"table": table_name, "issue": "Extra Column", "detail": col})
            
    # 3. Log to Metadata Table
    if violations:
        # Create a DataFrame from the list of dictionaries
        violations_df = spark.createDataFrame(violations)
        
        # Add a timestamp so we know when this happened
        violations_df = violations_df.withColumn("detected_at", F.current_timestamp())
        
        # Save to your metadata schema
        violations_df.write.mode("append").saveAsTable("globalmart.metadata.schema_violations")
        print(f"Found {len(violations)} violations for {table_name}. Logged to 'globalmart.metadata.schema_violations'.")
    else:
        print(f"{table_name} schema is clean and matches the contract!")

In [0]:
# 1. Define the 'Expected' contract (from your original CSV file)
# We use the original path to see what the data looked like BEFORE Task 1.5
orders_path = "/Volumes/globalmart/bronze/raw_data/olist_orders_dataset.csv"
expected_schema = spark.read.option("header", "true").option("inferSchema", "true").csv(orders_path).schema

# 2. Run the validation against the CURRENT evolved table
validate_schema(spark.table("globalmart.bronze.orders"), expected_schema, "orders")

# 3. View your violation log to prove the deliverable
display(spark.table("globalmart.metadata.schema_violations"))

In [0]:
%sql
SHOW TABLES IN globalmart.bronze;

In [0]:
from pyspark.sql import functions as F
df_try = spark.table("globalmart.bronze.orders").join(spark.table("globalmart.bronze.order_items"),"order_id")\
         .filter(F.col('order_status') == 'delivered')
df_try.explain('formatted')
        

In [0]:
orders_filtered = spark.table('globalmart.bronze.orders').filter('order_status == "delivered"')
df_join = orders_filtered.join(spark.table('globalmart.bronze.order_items'),"order_id")
df_join.explain("formatted")

In [0]:
# spark.conf.set('spark.sql.auto_broadcast_join_threshold', -1)
big_join = (spark.table("globalmart.bronze.products").join(spark.table("globalmart.bronze.product_category_name_translation").hint("merge"),"product_category_name"))
big_join.explain("formatted")


In [0]:
# BEST PRACTICE
from pyspark.sql.functions import broadcast
fast_join = spark.table("globalmart.bronze.products") \
    .join(broadcast(spark.table("globalmart.bronze.product_category_name_translation")), "product_category_name")

fast_join.explain() # Look for 'BroadcastHashJoin'

In [0]:
# Find skewed keys
from pyspark.sql import functions as F
spark.table("globalmart.bronze.order_items").groupBy("product_id").count().sort(F.desc("count")).show(5)

In [0]:
# Baseline Join
baseline_join = spark.table("globalmart.bronze.order_items").join(spark.table("globalmart.bronze.products"), "product_id")

# Document the plan
baseline_join.explain("formatted")
# Note the time
baseline_join.count()

In [0]:
from pyspark.sql import functions as F

# 1. Salt the Large Table (order_items)
# We add a random 'salt' from 0-9 to spread the 527 rows of your top product
items_salted = spark.table("globalmart.bronze.order_items") \
    .withColumn("salt", (F.rand() * 10).cast("int"))

# 2. Explode the Small Table (products)
# We must duplicate the products 10x so they can match any possible salt
products_exploded = spark.table("globalmart.bronze.products") \
    .withColumn("salts", F.array([F.lit(i) for i in range(10)])) \
    .withColumn("salt", F.explode("salts"))

# 3. The Salted Join
salted_join = items_salted.join(products_exploded, ["product_id", "salt"])

salted_join.explain("formatted")

In [0]:
# Problem: Find orders where any payment is > 100
explode_df = (spark.table("globalmart.bronze.payments_nested")
              .withColumn("p", F.explode("payment_details"))
              .filter("p.payment_value > 100")
              .select("order_id")
              .distinct())

explode_df.explain("formatted")

In [0]:
# Problem: Same result, but using HOF (exists)
hof_df = (spark.table("globalmart.bronze.payments_nested")
          .filter("exists(payment_details, x -> x.payment_value > 100)"))

hof_df.explain("formatted")

In [0]:
# Problem: Increase all payment values by 10% inside the array
transformed_df = (spark.table("globalmart.bronze.payments_nested")
                  .withColumn("increased_payments", 
                              F.expr("transform(payment_details, x -> struct(x.payment_value * 1.1 as payment_value, x.payment_type as payment_type))")))

transformed_df.show(5, truncate=False)